# Glass-Box Eval: Analysis

Compute and visualize agreement between LLM judge and human labels.

In [ ]:
import sys
sys.path.insert(0, '..')
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.metrics import build_metrics_table

RESULTS_DIR = Path('../results')
LABELS_FILE = Path('../data/annotations/labels.csv')

result_files = sorted(RESULTS_DIR.glob('*.csv'))
print(f'Found {len(result_files)} result file(s)')

In [ ]:
table = build_metrics_table(result_files, LABELS_FILE)
table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, metric in zip(axes, ['accuracy', 'kappa']):
    pivot = table.pivot(index='variant', columns='dimension', values=metric)
    sns.heatmap(pivot, annot=True, fmt='.2f', vmin=0, vmax=1, ax=ax, cmap='YlGn')
    ax.set_title(f'{metric.capitalize()} by Dimension and Prompt Variant')
    ax.set_ylabel('Prompt Variant')
    ax.set_xlabel('Dimension')

plt.tight_layout()
plt.savefig('../results/metrics_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(table))
labels = [f"{r.dimension}\n{r.variant}" for r in table.itertuples()]
ax.bar([i - 0.2 for i in x], table['accuracy'], width=0.4, label='Accuracy')
ax.bar([i + 0.2 for i in x], table['kappa'], width=0.4, label="Cohen's κ")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylim(-0.2, 1.1)
ax.set_ylabel('Score')
ax.set_title('LLM Judge Agreement with Human Labels')
ax.legend()
plt.tight_layout()
plt.savefig('../results/metrics_bar.png', dpi=150, bbox_inches='tight')
plt.show()